# 06 — Cancellation Prediction (Machine Learning)

Trains and compares three classifiers — Logistic Regression, Random
Forest, and XGBoost — to predict `is_cancelled` on `trips_ml_ready.csv`.

**Time-based split**: rows are sorted chronologically and the last 20% by
`request_datetime` become the test set. This simulates real deployment
(train on the past, predict the future) instead of an optimistic random
split that would let the model "see" patterns from dates after the ones
it's tested on.

In [1]:
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, classification_report,
)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as exc:  # pragma: no cover - environment dependent
    XGBOOST_AVAILABLE = False
    print(f"XGBoost is unavailable in this environment ({exc}).")
    print("macOS fix: `brew install libomp`, then re-run this notebook — "
          "XGBoost's PyPI wheel needs the system OpenMP runtime.")

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
IMAGES_DIR = Path("../images/ml")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

df = pd.read_csv(PROCESSED_DIR / "trips_ml_ready.csv", parse_dates=["request_datetime"])
df = df.sort_values("request_datetime").reset_index(drop=True)
print(df.shape, f"cancellation rate: {df.is_cancelled.mean():.3f}")

XGBoost is unavailable in this environment (
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/navdeeptaliyan/Downloads/files/Ride-Sharing-Analytics-Cancellation-Prediction/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <010DE1F1-B66F-31D2-8EDA-A08913D25DDA> /Users/navdeeptaliyan/Downloads/files/Ride-Sharing-Analytics-Cancellation-Prediction/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (n

## 1. Time-based train/test split (80/20)

In [2]:
split_idx = int(len(df) * 0.8)
split_date = df.iloc[split_idx].request_datetime
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()
print(f"Split date: {split_date}")
print(f"Train: {len(train_df):,} rows ({train_df.is_cancelled.mean():.3f} cancel rate)")
print(f"Test:  {len(test_df):,} rows ({test_df.is_cancelled.mean():.3f} cancel rate)")

Split date: 2025-08-31 14:59:00
Train: 36,000 rows (0.216 cancel rate)
Test:  9,000 rows (0.191 cancel rate)


## 2. Encode categoricals, keep train/test columns aligned

In [3]:
ID_COLS = ["trip_id", "rider_id", "driver_id", "request_datetime"]
TARGET = "is_cancelled"
CATEGORICAL = ["vehicle_type", "pickup_city", "drop_city", "payment_method",
               "rider_gender", "preferred_payment"]

combined = pd.get_dummies(df.drop(columns=ID_COLS + [TARGET]), columns=CATEGORICAL, drop_first=True)
feature_names = combined.columns.tolist()

X_train = combined.iloc[:split_idx].reset_index(drop=True)
X_test = combined.iloc[split_idx:].reset_index(drop=True)
y_train = train_df[TARGET].reset_index(drop=True)
y_test = test_df[TARGET].reset_index(drop=True)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

print(f"{len(feature_names)} features after encoding")

42 features after encoding


## 3. Train models

Deliberately **not** using `class_weight="balanced"` / `scale_pos_weight`
here: reweighting for the ~20% cancellation base rate would push
`predict_proba` away from true probabilities, which would break the
probability-threshold risk tiers in step 8 (everything would cluster into
"Medium"/"High" regardless of actual risk). Instead we train on the
natural class balance — keeping `predict_proba` calibrated to the real
base rate — and address the imbalance at the *decision threshold* instead
of the *loss function*: the risk tiers use business-chosen probability
cuts (0.30 / 0.60) rather than the default 0.5 classification threshold.

In [4]:
models = {}

log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
log_reg.fit(X_train_scaled, y_train)
models["Logistic Regression"] = (log_reg, X_test_scaled)

rf = RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=20,
    random_state=RANDOM_SEED, n_jobs=-1,
)
rf.fit(X_train, y_train)
models["Random Forest"] = (rf, X_test)

if XGBOOST_AVAILABLE:
    xgb_model = XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=-1,
    )
    xgb_model.fit(X_train, y_train)
    models["XGBoost"] = (xgb_model, X_test)

print(f"Trained: {list(models.keys())}")

Trained: ['Logistic Regression', 'Random Forest']


## 4. Evaluate

Metrics use a 0.30 decision threshold — matching the "Medium risk or
above" cut from the risk engine in step 8 — rather than the default 0.5,
since a missed cancellation is operationally costlier than a false alarm
and the base rate (~20%) means 0.5 is too conservative a bar for the
positive class.

In [5]:
DECISION_THRESHOLD = 0.30
comparison_rows = []
predictions_by_model = {}

for name, (model, X_te) in models.items():
    y_proba = model.predict_proba(X_te)[:, 1]
    y_pred = (y_proba >= DECISION_THRESHOLD).astype(int)

    comparison_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })
    predictions_by_model[name] = y_proba
    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred, target_names=["Completed", "Cancelled"]))

model_comparison = pd.DataFrame(comparison_rows).sort_values("f1_score", ascending=False)
model_comparison.to_csv(PROCESSED_DIR / "model_comparison.csv", index=False)
print(model_comparison.round(4))


--- Logistic Regression ---
              precision    recall  f1-score   support

   Completed       0.83      0.91      0.87      7278
   Cancelled       0.35      0.21      0.27      1722

    accuracy                           0.77      9000
   macro avg       0.59      0.56      0.57      9000
weighted avg       0.74      0.77      0.75      9000


--- Random Forest ---
              precision    recall  f1-score   support

   Completed       0.84      0.94      0.89      7278
   Cancelled       0.48      0.24      0.32      1722

    accuracy                           0.80      9000
   macro avg       0.66      0.59      0.60      9000
weighted avg       0.77      0.80      0.78      9000

                 model  accuracy  precision  recall  f1_score  roc_auc
1        Random Forest    0.8040     0.4755  0.2369    0.3163   0.6771
0  Logistic Regression    0.7736     0.3504  0.2149    0.2664   0.6161


## 5. Select the best model

Recall and F1 matter more than raw accuracy here — missing a high-risk
cancellation is operationally costlier than a false alarm.

In [6]:
best_model_name = model_comparison.iloc[0]["model"]
best_model, best_X_test = models[best_model_name]
print(f"Best model by F1-score: {best_model_name}")

model_filename = "cancellation_xgboost.pkl" if best_model_name == "XGBoost" else \
    f"cancellation_{best_model_name.lower().replace(' ', '_')}.pkl"
joblib.dump(best_model, MODELS_DIR / model_filename)
print(f"Saved best model -> models/{model_filename}")

if not XGBOOST_AVAILABLE:
    print("NOTE: XGBoost wasn't available in this run, so the best model above is not "
          "XGBoost. Install libomp and re-run to get the XGBoost comparison + "
          "models/cancellation_xgboost.pkl the README describes.")

Best model by F1-score: Random Forest
Saved best model -> models/cancellation_random_forest.pkl
NOTE: XGBoost wasn't available in this run, so the best model above is not XGBoost. Install libomp and re-run to get the XGBoost comparison + models/cancellation_xgboost.pkl the README describes.


## 6. ROC curves

In [7]:
plt.figure(figsize=(8, 6))
for name, y_proba in predictions_by_model.items():
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Cancellation Prediction Models")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "01_roc_curves.png", dpi=140, bbox_inches="tight")
plt.close()

## 7. Confusion matrix — best model

In [8]:
best_proba = predictions_by_model[best_model_name]
best_pred = (best_proba >= DECISION_THRESHOLD).astype(int)
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["Completed", "Cancelled"])
plt.yticks([0, 1], ["Completed", "Cancelled"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center",
                  color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "02_confusion_matrix.png", dpi=140, bbox_inches="tight")
plt.close()

## 8. Cancellation risk scoring

Converts predicted probability into an operational risk tier.

In [9]:
def risk_tier(p):
    if p < 0.30:
        return "Low"
    elif p < 0.60:
        return "Medium"
    return "High"

predictions_out = test_df[["trip_id", "rider_id", "driver_id", "request_datetime"]].copy()
predictions_out["actual_cancelled"] = y_test.values
predictions_out["predicted_cancel_probability"] = best_proba.round(4)
predictions_out["predicted_cancelled"] = best_pred
predictions_out["risk_tier"] = predictions_out.predicted_cancel_probability.apply(risk_tier)
predictions_out["model_used"] = best_model_name

predictions_out.to_csv(PROCESSED_DIR / "cancellation_predictions.csv", index=False)
print(predictions_out.risk_tier.value_counts())
print(f"\nSaved {len(predictions_out):,} predictions -> data/processed/cancellation_predictions.csv")

risk_tier
Low       8141
Medium     696
High       163
Name: count, dtype: int64

Saved 9,000 predictions -> data/processed/cancellation_predictions.csv


## 9. Export live-demo artifacts

`streamlit_app/app.py` loads the saved model and needs to reproduce this
notebook's exact preprocessing (same one-hot columns, in the same order,
same scaler) without re-running the notebook. Everything it needs is
exported here, computed from real training data rather than hardcoded —
if you re-run this notebook after adding a new city or vehicle type, the
live demo picks it up automatically.

In [10]:
import json

RAW_FEATURE_COLUMNS = df.drop(columns=ID_COLS + [TARGET]).columns.tolist()
NUMERIC_COLUMNS = [c for c in RAW_FEATURE_COLUMNS if c not in CATEGORICAL]

with open(MODELS_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_names, f, indent=2)

with open(MODELS_DIR / "raw_feature_columns.json", "w") as f:
    json.dump({"all": RAW_FEATURE_COLUMNS, "categorical": CATEGORICAL, "numeric": NUMERIC_COLUMNS}, f, indent=2)

categorical_options = {col: sorted(df[col].dropna().unique().tolist()) for col in CATEGORICAL}
with open(MODELS_DIR / "categorical_options.json", "w") as f:
    json.dump(categorical_options, f, indent=2)

feature_ranges = {
    col: {
        "min": float(df[col].min()),
        "median": float(df[col].median()),
        "max": float(df[col].max()),
    }
    for col in NUMERIC_COLUMNS
}
with open(MODELS_DIR / "feature_ranges.json", "w") as f:
    json.dump(feature_ranges, f, indent=2)

base_fare_by_vehicle_type = df.groupby("vehicle_type")["base_fare"].median().round(2).to_dict()
with open(MODELS_DIR / "base_fare_by_vehicle_type.json", "w") as f:
    json.dump(base_fare_by_vehicle_type, f, indent=2)

joblib.dump(scaler, MODELS_DIR / "scaler.pkl")

model_metadata = {
    "best_model_name": best_model_name,
    "model_filename": model_filename,
    "is_linear_model": best_model_name == "Logistic Regression",
    "decision_threshold": DECISION_THRESHOLD,
    "risk_tier_cuts": {"low_max": 0.30, "medium_max": 0.60},
    "xgboost_available_at_training_time": XGBOOST_AVAILABLE,
}
with open(MODELS_DIR / "model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Saved live-demo artifacts to models/:")
print("  feature_columns.json, raw_feature_columns.json, categorical_options.json,")
print("  feature_ranges.json, base_fare_by_vehicle_type.json, scaler.pkl, model_metadata.json")

Saved live-demo artifacts to models/:
  feature_columns.json, raw_feature_columns.json, categorical_options.json,
  feature_ranges.json, base_fare_by_vehicle_type.json, scaler.pkl, model_metadata.json
